# 🤖 Notebook 04 — Model Training & Evaluation

**Input:** `data/processed/features.csv`  
**Output:** `models/best_model.pkl` + `models/random_forest.pkl`  

---

## Models Compared

| Model | Strength | Weakness |
|---|---|---|
| Decision Tree | Fast, interpretable | Overfits |
| **Random Forest** | Best baseline for IMU | Slower |
| SVM | Good with scaled features | Slow on large data |
| KNN | Simple | Slow at inference |
| XGBoost | Top accuracy | Needs tuning |

**Expected winner: Random Forest or XGBoost.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.feature_engineering import apply_sliding_window, split_dataset
from src.train_model import (
    get_models, evaluate_model, plot_comparison,
    plot_confusion_matrix, plot_feature_importance, save_best_model
)

print('All imports OK')

In [ ]:
# ── Load features (or regenerate) ────────────────────────────────
feat_path = '../data/processed/features.csv'
raw_path  = '../data/synthetic/helmet_imu_raw.csv'

if os.path.exists(feat_path):
    feature_df = pd.read_csv(feat_path)
    print(f'Loaded features: {feature_df.shape}')
else:
    print('Feature file not found. Regenerating...')
    df = pd.read_csv(raw_path)
    feature_df = apply_sliding_window(df, verbose=True)
    os.makedirs('../data/processed', exist_ok=True)
    feature_df.to_csv(feat_path, index=False)

display(feature_df.head(3))

In [ ]:
# ── Train / Test split ────────────────────────────────────────────
X_train, X_test, y_train, y_test = split_dataset(feature_df, test_size=0.2, seed=42)
feature_cols = X_train.columns.tolist()

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'Features: {len(feature_cols)}')

In [ ]:
# ── Train all models ──────────────────────────────────────────────
# This cell will take a few minutes depending on your machine
models  = get_models()
results = []

for name, model in models.items():
    print(f'\nTraining: {name}...')
    res = evaluate_model(model, X_train, X_test, y_train, y_test, name)
    results.append(res)

print('\n✅ All models trained!')

In [ ]:
# ── Model comparison bar chart ────────────────────────────────────
plot_comparison(results)

In [ ]:
# ── Confusion matrix — best model ────────────────────────────────
best_result = max(results, key=lambda r: r['f1_macro'])
print(f'Best model: {best_result["model_name"]} (F1 macro = {best_result["f1_macro"]:.4f})')
plot_confusion_matrix(best_result)

In [ ]:
# ── Confusion matrix — Random Forest (always show for comparison) ─
rf_result = next((r for r in results if 'Random Forest' in r['model_name']), None)
if rf_result:
    print('Random Forest confusion matrix:')
    plot_confusion_matrix(rf_result)

In [ ]:
# ── Feature importance ────────────────────────────────────────────
if rf_result:
    plot_feature_importance(rf_result['model'], feature_cols, top_n=20, title='Random Forest')

plot_feature_importance(best_result['model'], feature_cols, top_n=20, title=best_result['model_name'])

In [ ]:
# ── Save models ───────────────────────────────────────────────────
os.makedirs('../models', exist_ok=True)
best = save_best_model(results, feature_cols, save_dir='../models')

In [ ]:
# ── Summary table ─────────────────────────────────────────────────
summary = pd.DataFrame([{
    'Model'      : r['model_name'],
    'Accuracy'   : f"{r['accuracy']:.4f}",
    'F1 Macro'   : f"{r['f1_macro']:.4f}",
    'F1 Weighted': f"{r['f1_weighted']:.4f}"
} for r in results]).sort_values('F1 Macro', ascending=False).reset_index(drop=True)

print('\n=== FINAL MODEL COMPARISON ===')
display(summary)
print(f'\n✅ Best: {best["model_name"]} saved to models/')
print('\nNext: Run src/predict.py to simulate live prediction!')